In [1]:
# =============================================================================
# CELL 1 — DATA FIREWALL
# Pre-specified constants. Do not modify after OOS evaluation begins.
# =============================================================================
import sys
from pathlib import Path

# Auto-detect project root (works from any working directory)
_nb_root = Path.cwd()
for _candidate in [_nb_root, _nb_root.parent, _nb_root.parent.parent]:
    if (_candidate / 'src').is_dir():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# ---------------------------------------------------------------------------
# Pre-specified constants (FIXED — do not derive from test data)
# ---------------------------------------------------------------------------
REAL_RATE_THRESHOLD = 0.0   # regime=1 when real_rate < 0; fixed prior, not data-derived
TRAIN_END   = '2007-12-31'
TEST_START  = '2008-03-31'
CITIES      = ['austin', 'phoenix', 'las_vegas', 'miami']
HORIZON     = 20            # quarters for crash look-ahead
DROP_THRESH = 0.90          # price must stay above 90% of current level

# ---------------------------------------------------------------------------
# Assertions — must pass before any test data is loaded
# ---------------------------------------------------------------------------
import pandas as pd
assert REAL_RATE_THRESHOLD == 0.0, 'REAL_RATE_THRESHOLD must be 0.0 (pre-specified prior)'
assert pd.Timestamp(TEST_START) > pd.Timestamp(TRAIN_END), 'Test must start after train end'
assert 'DTI_FRAGILITY_CUTOFF' not in dir(), 'DTI must not appear in regime-only notebook'

print('=== DATA FIREWALL PASSED ===')
print(f'REAL_RATE_THRESHOLD : {REAL_RATE_THRESHOLD}')
print(f'Train window        : 1990Q1 – 2007Q4')
print(f'Test window         : 2008Q1 – latest')
print(f'Cities              : {CITIES}')
print(f'Regime rule         : real_rate < {REAL_RATE_THRESHOLD} → regime=1 (accommodative)')
print(f'Overlay rule        : invest only when regime == 1 (accommodative)')
print(f'No DTI cutoff used  : regime-only test')

=== DATA FIREWALL PASSED ===
REAL_RATE_THRESHOLD : 0.0
Train window        : 1990Q1 – 2007Q4
Test window         : 2008Q1 – latest
Cities              : ['austin', 'phoenix', 'las_vegas', 'miami']
Regime rule         : real_rate < 0.0 → regime=1 (accommodative)
Overlay rule        : invest only when regime == 1 (accommodative)
No DTI cutoff used  : regime-only test


In [2]:
# =============================================================================
# CELL 2 — DATA LOAD
# Load HPI panel + national real rate; build crash labels; split train/test.
# =============================================================================
import numpy as np
import requests
from io import StringIO

from src.loaders.fred import load_fred_hpi_panel
from src.evaluation.regime import assign_fragility_regime

# --- HPI panel (four cities, FRED FHFA quarterly) ---------------------------
hpi = load_fred_hpi_panel(cities=CITIES, start='1982-01-01')

# --- National real rate (FRED REAINTRATREARAT10Y, monthly → quarterly) ------
url = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=REAINTRATREARAT10Y'
rate_raw = pd.read_csv(StringIO(requests.get(url, timeout=30).text))
rate_raw.columns = ['date', 'real_rate']
rate_raw['date'] = pd.to_datetime(rate_raw['date'])
rate_raw['real_rate'] = pd.to_numeric(rate_raw['real_rate'], errors='coerce')
# QS to match FRED HPI date convention (Jan 1, Apr 1, Jul 1, Oct 1)
rate_q = (
    rate_raw.set_index('date')
    .resample('QS').mean()
    .reset_index()
    .dropna(subset=['real_rate'])
)

# --- Merge & regime assignment ----------------------------------------------
panel = pd.merge(hpi, rate_q[['date', 'real_rate']], on='date', how='inner')
panel = assign_fragility_regime(panel, threshold=REAL_RATE_THRESHOLD)

# Quarterly log return (per city, time-sorted)
panel['ret_1q'] = panel.groupby('region')['price'].transform(
    lambda s: np.log(s).diff()
)

# --- Crash label (20-quarter look-ahead, >10% drawdown) --------------------
def add_crash_label(g):
    g = g.sort_values('date').copy()
    prices = g['price'].to_numpy()
    n = len(prices)
    y = [np.nan] * n
    for t in range(n):
        if t + HORIZON < n:
            y[t] = 1 if prices[t+1:t+HORIZON+1].min() / prices[t] < DROP_THRESH else 0
    g['y'] = y
    return g

frames = [add_crash_label(g) for _, g in panel.groupby('region')]
panel = pd.concat(frames, ignore_index=True).sort_values(['region', 'date']).reset_index(drop=True)

# --- Train / test split -----------------------------------------------------
train = panel[panel['date'] <= TRAIN_END].copy()
test  = panel[panel['date'] >= TEST_START].copy()

# Firewall assertion: no test leakage
assert test['date'].min() >= pd.Timestamp(TEST_START), 'Test data leaks into train window'
assert train['date'].max() <= pd.Timestamp(TRAIN_END), 'Train data bleeds into test window'

print(f'Panel shape : {panel.shape}')
print(f'Columns     : {list(panel.columns)}')
print()
for city in CITIES:
    tr = train[train['region'] == city]
    te = test[test['region'] == city]
    print(f'{city:12s}: train={len(tr)}, test={len(te)}, '
          f'crashes_train={tr["y"].sum():.0f}, crashes_test={te["y"].sum():.0f}')

print()
print('Regime breakdown (test):')
print(test.groupby(['region', 'regime']).size().unstack(fill_value=0))
print(f'\nReal rate (train) range: {train["real_rate"].min():.2f}% to {train["real_rate"].max():.2f}%')
print(f'Real rate (test)  range: {test["real_rate"].min():.2f}% to {test["real_rate"].max():.2f}%')

Panel shape : (704, 9)
Columns     : ['region', 'date', 'price', 'source_label', 'data_weight', 'real_rate', 'regime', 'ret_1q', 'y']

austin      : train=104, test=71, crashes_train=17, crashes_test=0
phoenix     : train=104, test=71, crashes_train=12, crashes_test=11
las_vegas   : train=104, test=71, crashes_train=21, crashes_test=11
miami       : train=104, test=71, crashes_train=13, crashes_test=5

Regime breakdown (test):
regime      0  1
region          
austin     64  7
las_vegas  64  7
miami      64  7
phoenix    64  7

Real rate (train) range: 1.20% to 7.47%
Real rate (test)  range: -0.37% to 2.01%


In [3]:
# =============================================================================
# TABLE 1 — Cell counts (regime) for train and test periods, per city + pooled
# =============================================================================
try:
    display
except NameError:
    def display(df): print(df.to_string(index=False))

def make_cell_counts(df, period_label, region_label='all'):
    """Return a DataFrame with cell counts by regime for one period/region."""
    rows = []
    labeled = df.dropna(subset=['y'])
    for regime_val, regime_name in [(0, 'adverse (regime=0)'), (1, 'accommodative (regime=1)')]:
        mask = df['regime'] == regime_val
        lmask = labeled['regime'] == regime_val
        n_obs     = mask.sum()
        n_labeled = lmask.sum()
        n_crash   = labeled.loc[lmask, 'y'].sum()
        rows.append({
            'period':   period_label,
            'region':   region_label,
            'regime':   regime_name,
            'n_obs':    int(n_obs),
            'n_labeled': int(n_labeled),
            'n_crash':  int(n_crash),
            'low_n':    n_labeled < 10,
        })
    return pd.DataFrame(rows)

table1_rows = []

# Per-city
for city in CITIES:
    for split_df, label in [(train, 'Train (1990Q1–2007Q4)'), (test, 'Test (2008Q1–latest)')]:
        sub = split_df[split_df['region'] == city]
        table1_rows.append(make_cell_counts(sub, label, city))

# Pooled
for split_df, label in [(train, 'Train (1990Q1–2007Q4)'), (test, 'Test (2008Q1–latest)')]:
    table1_rows.append(make_cell_counts(split_df, label, 'POOLED'))

table1 = pd.concat(table1_rows, ignore_index=True)
print('TABLE 1: Cell counts by regime')
print('=' * 80)
display(table1[['period', 'region', 'regime', 'n_obs', 'n_labeled', 'n_crash', 'low_n']])

TABLE 1: Cell counts by regime


,period,region,regime,n_obs,n_labeled,n_crash,low_n
0,Train (1990Q1–2007Q4),austin,adverse (regime=0),104,104,17,False
1,Train (1990Q1–2007Q4),austin,accommodative (regime=1),0,0,0,True
2,Test (2008Q1–latest),austin,adverse (regime=0),64,46,0,False
3,Test (2008Q1–latest),austin,accommodative (regime=1),7,5,0,True
4,Train (1990Q1–2007Q4),phoenix,adverse (regime=0),104,104,12,False
5,Train (1990Q1–2007Q4),phoenix,accommodative (regime=1),0,0,0,True
6,Test (2008Q1–latest),phoenix,adverse (regime=0),64,46,11,False
7,Test (2008Q1–latest),phoenix,accommodative (regime=1),7,5,0,True
8,Train (1990Q1–2007Q4),las_vegas,adverse (regime=0),104,104,21,False
9,Train (1990Q1–2007Q4),las_vegas,accommodative (regime=1),0,0,0,True


In [4]:
# =============================================================================
# TABLE 2 — Conditional crash frequency with Wilson 95% CI, per city + pooled
# =============================================================================
try:
    display
except NameError:
    def display(df): print(df.to_string(index=False))

from src.backtests.evaluation import bootstrap_ci

def wilson_ci(k, n, z=1.96):
    """Wilson score interval for a proportion k/n."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (max(0.0, centre - half), min(1.0, centre + half))


def make_freq_table(df, period_label, region_label='all'):
    labeled = df.dropna(subset=['y'])
    rows = []
    for regime_val, regime_name in [(0, 'adverse'), (1, 'accommodative')]:
        sub = labeled[labeled['regime'] == regime_val]
        n = len(sub)
        k = int(sub['y'].sum())
        freq = k / n if n > 0 else np.nan
        lo, hi = wilson_ci(k, n)
        # Weighted bootstrap CI on crash indicator
        if n >= 2:
            blo, bhi = bootstrap_ci(
                sub['y'],
                stat=np.mean,
                weights=sub['data_weight'],
                B=2000,
                rng=np.random.default_rng(42),
            )
        else:
            blo, bhi = np.nan, np.nan
        rows.append({
            'period':     period_label,
            'region':     region_label,
            'regime':     regime_name,
            'n':          n,
            'crashes':    k,
            'freq':       round(freq, 4) if not np.isnan(freq) else np.nan,
            'wilson_lo':  round(lo, 4),
            'wilson_hi':  round(hi, 4),
            'boot_lo':    round(blo, 4) if not np.isnan(blo) else np.nan,
            'boot_hi':    round(bhi, 4) if not np.isnan(bhi) else np.nan,
        })
    return pd.DataFrame(rows)


table2_rows = []

# Per-city (test only — train shown separately)
for city in CITIES:
    for split_df, label in [(train, 'Train'), (test, 'Test')]:
        sub = split_df[split_df['region'] == city]
        table2_rows.append(make_freq_table(sub, label, city))

# Pooled
for split_df, label in [(train, 'Train'), (test, 'Test')]:
    table2_rows.append(make_freq_table(split_df, label, 'POOLED'))

table2 = pd.concat(table2_rows, ignore_index=True)
print('TABLE 2: Conditional crash frequency with Wilson 95% CI')
print('=' * 80)
display(table2)

# --- Pre-specified success criterion check (pooled test window) --------------
test_pooled = table2[(table2['period'] == 'Test') & (table2['region'] == 'POOLED')]
adverse_row = test_pooled[test_pooled['regime'] == 'adverse'].iloc[0]
accom_row   = test_pooled[test_pooled['regime'] == 'accommodative'].iloc[0]

ci_non_overlap = adverse_row['wilson_lo'] > accom_row['wilson_hi'] or \
                 accom_row['wilson_lo'] > adverse_row['wilson_hi']
adverse_higher = adverse_row['freq'] > accom_row['freq']

print()
print('SUCCESS CRITERION: adverse crash freq > accommodative crash freq (non-overlapping Wilson CIs)')
print(f"  Adverse   : freq={adverse_row['freq']:.4f}  CI=[{adverse_row['wilson_lo']:.4f}, {adverse_row['wilson_hi']:.4f}]")
print(f"  Accomm.   : freq={accom_row['freq']:.4f}  CI=[{accom_row['wilson_lo']:.4f}, {accom_row['wilson_hi']:.4f}]")
print(f"  Direction : {'✓ adverse > accommodative' if adverse_higher else '✗ NOT adverse > accommodative'}")
print(f"  CI overlap: {'✓ non-overlapping' if ci_non_overlap else '✗ overlapping — criterion NOT met'}")

TABLE 2: Conditional crash frequency with Wilson 95% CI


,period,region,regime,n,crashes,freq,wilson_lo,wilson_hi,boot_lo,boot_hi
0,Train,austin,adverse,104,17,0.1635,0.1046,0.2463,0.0962,0.2404
1,Train,austin,accommodative,0,0,NaN,NaN,NaN,NaN,NaN
2,Test,austin,adverse,46,0,0.0000,0.0000,0.0771,0.0000,0.0000
3,Test,austin,accommodative,5,0,0.0000,0.0000,0.4345,0.0000,0.0000
4,Train,phoenix,adverse,104,12,0.1154,0.0672,0.1909,0.0577,0.1827
5,Train,phoenix,accommodative,0,0,NaN,NaN,NaN,NaN,NaN
6,Test,phoenix,adverse,46,11,0.2391,0.1391,0.3794,0.1087,0.3696
7,Test,phoenix,accommodative,5,0,0.0000,0.0000,0.4345,0.0000,0.0000
8,Train,las_vegas,adverse,104,21,0.2019,0.1360,0.2890,0.1344,0.2788
9,Train,las_vegas,accommodative,0,0,NaN,NaN,NaN,NaN,NaN



SUCCESS CRITERION: adverse crash freq > accommodative crash freq (non-overlapping Wilson CIs)
  Adverse   : freq=0.1467  CI=[0.1028, 0.2051]
  Accomm.   : freq=0.0000  CI=[0.0000, 0.1611]
  Direction : ✓ adverse > accommodative
  CI overlap: ✗ overlapping — criterion NOT met


In [5]:
# =============================================================================
# TABLE 3 — Strategy comparison: always-in, regime overlay, always-out
# Test window only. Per city + pooled.
# =============================================================================
try:
    display
except NameError:
    def display(df): print(df.to_string(index=False))

from src.backtests.evaluation import summarize

def strategy_metrics(ret_series, name):
    """Annualized metrics from quarterly log-return series."""
    s = summarize(ret_series.dropna(), name=name)
    ann_factor = 4
    time_in = (ret_series.dropna() != 0).mean()
    vol = s['vol']
    sharpe_ann = round(s['mean'] / vol * np.sqrt(ann_factor), 4) if vol and vol > 0 else np.nan
    return {
        'strategy':      s['name'],
        'ann_return':    round(s['mean'] * ann_factor, 4),
        'ann_vol':       round(vol * np.sqrt(ann_factor), 4) if vol else np.nan,
        'sharpe':        sharpe_ann,
        'max_drawdown':  round(s['maxdd'], 4),
        'time_in_mkt':   round(float(time_in), 4),
        'n_qtrs':        s['n'],
    }


def compute_strategies(sub_test):
    """Return list of metric dicts for always_in, regime_overlay, always_out."""
    sub = sub_test.sort_values('date').copy()
    ret = sub['ret_1q']
    # always-in: full quarterly return
    always_in  = ret
    # regime overlay: invest (take ret) only when regime=1 (accommodative), else 0
    overlay    = ret.where(sub['regime'] == 1, other=0.0)
    # always-out: zero return every quarter
    always_out = pd.Series(0.0, index=sub.index)
    return [
        strategy_metrics(always_in,  'always_in'),
        strategy_metrics(overlay,    'regime_overlay'),
        strategy_metrics(always_out, 'always_out'),
    ]


table3_rows = []

# Per-city
for city in CITIES:
    sub = test[test['region'] == city]
    for m in compute_strategies(sub):
        m['region'] = city
        table3_rows.append(m)

# Pooled (stack all cities, sort by date — regime applies per-row)
for m in compute_strategies(test.copy()):
    m['region'] = 'POOLED'
    table3_rows.append(m)

table3 = pd.DataFrame(table3_rows)[[
    'region', 'strategy', 'ann_return', 'ann_vol', 'sharpe', 'max_drawdown', 'time_in_mkt', 'n_qtrs'
]]

print('TABLE 3: Strategy comparison (test window, annualized)')
print('=' * 80)
display(table3)

# --- Pre-specified success criterion checks ----------------------------------
pooled3 = table3[table3['region'] == 'POOLED']
ai  = pooled3[pooled3['strategy'] == 'always_in'].iloc[0]
ro  = pooled3[pooled3['strategy'] == 'regime_overlay'].iloc[0]
ao  = pooled3[pooled3['strategy'] == 'always_out'].iloc[0]

print()
print('SUCCESS CRITERION 1: Regime overlay max drawdown < always-in max drawdown (pooled)')
print(f"  always_in  maxdd = {ai['max_drawdown']:.4f}")
print(f"  overlay    maxdd = {ro['max_drawdown']:.4f}")
print(f"  Result: {'✓ PASS' if ro['max_drawdown'] > ai['max_drawdown'] else '✗ FAIL'}")

print()
print('FAILURE CRITERION CHECK: Always-out outperforms overlay on risk-adjusted basis (pooled)')
print(f"  always_out sharpe = {ao['sharpe']}")
print(f"  overlay    sharpe = {ro['sharpe']:.4f}")
ao_sharpe = ao['sharpe'] if not np.isnan(float(ao['sharpe'])) else 0.0
print(f"  Result: {'✗ FAILURE CRITERION MET — always-out wins on sharpe' if ao_sharpe > ro['sharpe'] else '✓ Failure criterion not triggered'}")

TABLE 3: Strategy comparison (test window, annualized)


,region,strategy,ann_return,ann_vol,sharpe,max_drawdown,time_in_mkt,n_qtrs
0,austin,always_in,0.0532,0.0499,1.0661,-0.1183,1.0000,71
1,austin,regime_overlay,0.0135,0.0270,0.4986,0.0000,0.0986,71
2,austin,always_out,0.0000,NaN,NaN,0.0000,0.0000,71
3,phoenix,always_in,0.0381,0.0708,0.5385,-0.4146,1.0000,71
4,phoenix,regime_overlay,0.0165,0.0278,0.5918,0.0000,0.0986,71
5,phoenix,always_out,0.0000,NaN,NaN,0.0000,0.0000,71
6,las_vegas,always_in,0.0309,0.0849,0.3642,-0.4876,0.9859,71
7,las_vegas,regime_overlay,0.0120,0.0226,0.5285,0.0000,0.0986,71
8,las_vegas,always_out,0.0000,NaN,NaN,0.0000,0.0000,71
9,miami,always_in,0.0413,0.0682,0.6052,-0.3821,1.0000,71



SUCCESS CRITERION 1: Regime overlay max drawdown < always-in max drawdown (pooled)
  always_in  maxdd = -0.8596
  overlay    maxdd = 0.0000
  Result: ✓ PASS

FAILURE CRITERION CHECK: Always-out outperforms overlay on risk-adjusted basis (pooled)
  always_out sharpe = nan
  overlay    sharpe = 0.5435
  Result: ✓ Failure criterion not triggered


In [6]:
# =============================================================================
# FIGURE 1 — Equity curves + regime transitions, per city (2×2) + pooled
# =============================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Resolve output dir relative to project root (works in notebook and script contexts)
_here = Path(__file__).resolve().parent if '__file__' in dir() else Path.cwd()
_root = _here
for _cand in [_here, _here.parent, _here.parent.parent]:
    if (_cand / 'src').is_dir():
        _root = _cand
        break
OUT_DIR = _root / 'outputs' / 'oos'
OUT_DIR.mkdir(parents=True, exist_ok=True)


def plot_equity_curves(sub_test, ax, title):
    """Plot always-in, regime overlay, always-out equity curves on ax."""
    sub = sub_test.sort_values('date').copy().reset_index(drop=True)
    dates = sub['date']
    ret   = sub['ret_1q'].fillna(0.0)
    regime = sub['regime']

    always_in  = np.exp(ret.cumsum())
    overlay_r  = ret.where(regime == 1, other=0.0)
    overlay    = np.exp(overlay_r.cumsum())
    always_out = pd.Series(1.0, index=sub.index)

    ax.plot(dates, always_in,  color='steelblue',  lw=1.5, label='Always-in')
    ax.plot(dates, overlay,    color='darkorange',  lw=1.5, label='Regime overlay (accom. only)')
    ax.plot(dates, always_out, color='gray',        lw=1.0, ls='--', label='Always-out')

    # Shade accommodative regime periods (regime=1)
    in_accom = False
    start_idx = None
    for i, (d, r) in enumerate(zip(dates, regime)):
        if r == 1 and not in_accom:
            start_idx = i
            in_accom = True
        elif r != 1 and in_accom:
            ax.axvspan(dates.iloc[start_idx], dates.iloc[i-1],
                       alpha=0.12, color='green', label='_nolegend_')
            in_accom = False
    if in_accom:
        ax.axvspan(dates.iloc[start_idx], dates.iloc[-1],
                   alpha=0.12, color='green', label='_nolegend_')

    # GFC annotation
    gfc_date = pd.Timestamp('2008-09-01')
    if dates.min() <= gfc_date <= dates.max():
        ax.axvline(gfc_date, color='red', lw=0.8, ls=':', alpha=0.7)
        ax.text(gfc_date, ax.get_ylim()[1] * 0.97, 'GFC', fontsize=6,
                color='red', ha='left', va='top')

    # 2022 rate-rise annotation
    rate_rise = pd.Timestamp('2022-01-01')
    if dates.min() <= rate_rise <= dates.max():
        ax.axvline(rate_rise, color='purple', lw=0.8, ls=':', alpha=0.7)
        ax.text(rate_rise, ax.get_ylim()[1] * 0.97, '2022', fontsize=6,
                color='purple', ha='left', va='top')

    ax.set_title(title, fontsize=9)
    ax.set_ylabel('Equity (base=1.0)', fontsize=7)
    ax.tick_params(labelsize=6)
    ax.grid(True, alpha=0.3)


# --- 2×2 grid for four cities ------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Figure 1: Equity Curves by City — Regime-Only Overlay (Test Window 2008Q1+)',
             fontsize=11, fontweight='bold')

for ax, city in zip(axes.flat, CITIES):
    sub = test[test['region'] == city]
    plot_equity_curves(sub, ax, city.replace('_', ' ').title())

# Shared legend
handles, labels = axes.flat[0].get_legend_handles_labels()
green_patch = mpatches.Patch(color='green', alpha=0.25, label='Accommodative regime (regime=1)')
fig.legend(handles + [green_patch], labels + ['Accommodative regime (regime=1)'],
           loc='lower center', ncol=4, fontsize=8, bbox_to_anchor=(0.5, 0.01))

plt.tight_layout(rect=[0, 0.06, 1, 1])
fig1_path = OUT_DIR / 'crossmarket_regime_equity_curves.png'
plt.savefig(fig1_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure 1 saved → {fig1_path}')


# --- Pooled equity curve -----------------------------------------------------
fig2, ax2 = plt.subplots(figsize=(10, 5))
fig2.suptitle('Figure 1 (Pooled): Equity Curves — Regime-Only Overlay (Test Window 2008Q1+)',
              fontsize=11, fontweight='bold')
plot_equity_curves(test.copy(), ax2, 'All Cities Pooled')
ax2.legend(fontsize=8)
plt.tight_layout()
fig1p_path = OUT_DIR / 'crossmarket_regime_equity_curves_pooled.png'
plt.savefig(fig1p_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure 1 (pooled) saved → {fig1p_path}')

Figure 1 saved → /Users/axl/leviathan-system/outputs/oos/crossmarket_regime_equity_curves.png
Figure 1 (pooled) saved → /Users/axl/leviathan-system/outputs/oos/crossmarket_regime_equity_curves_pooled.png


/var/folders/dg/2kv5bj0j5d9c8wklbylhzg800000gn/T/ipykernel_27558/2949781524.py:89: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/dg/2kv5bj0j5d9c8wklbylhzg800000gn/T/ipykernel_27558/2949781524.py:102: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
